# FAISS for TEXT - Quick Start
This notebook is a companion of chapter 2 of the "Domain Specific LLms in Action" book, author Guglielmo Iozzia, [Manning Publications](https://www.manning.com/), 2024.  
The code in this notebook is to introduce readers to the [FAISS](https://faiss.ai/index.html) library. No hardware acceleration required to execute all the code cells.  

Install the missing required packages in the Colab VM. Only FAISS for CPU is missing. Then downgrade the now available [SentenceTransformers](https://www.sbert.net/) release to 4.1.0 for compatibility with the code in this notebook.

In [1]:
!pip install faiss-cpu
!pip uninstall sentence-transformers
!pip install sentence-transformers==4.1.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 137.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 180.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 194.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 176.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.2/801.2 kB 72.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [sentence-transformers]ence-transformers]


Import the necessary packages/classes.

In [3]:
"""Module to cluster embeddings and create indices."""
import faiss

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

Set the data corpus for this example and put it into a Pandas DataFrame.

In [6]:
data = [['His secret identity is Peter Parker', 'spiderman'],
        ['A businessman and engineer who ' +
         'runs the company Stark Industries',
         'ironman'],
        ['Superhuman spider-powers and abilities ' +
         'after being bitten by a radioactive spider',
         'spiderman'],
        ['A frail man enhanced to the peak of human ' +
         'physical perfection by an experimental super-soldier serum', 'captainamerica']
        ]
df = pd.DataFrame(data, columns = ['text', 'context'])
df.head(10)

,text,context
0,His secret identity is Peter Parker,spiderman
1,A businessman and engineer who runs the compan...,ironman
2,Superhuman spider-powers and abilities after b...,spiderman
3,A frail man enhanced to the peak of human phys...,captainamerica


Get embeddings from the data corpus, generate a FAISS index and add the embeddings to it (after normalization).  
To make the code in the cell below compatible with SentenceTransformers release 5.0+, simple replace the line  
```vectors = encoder.encode(text)```  
with  
```vectors = encoder.encode(text.to_list())```

In [8]:
text = df['text']
encoder = SentenceTransformer("paraphrase-mpnet-base-v2")
vectors = encoder.encode(text)
vector_dimension = vectors.shape[1]
print('vector_dimension===>', vector_dimension)
l2_index = faiss.IndexFlatL2(vector_dimension)
faiss.normalize_L2(vectors)
l2_index.add(vectors)

vector_dimension===> 768


Prepare a search text to be used for similarity search with FAISS on the generated index.

In [28]:
# search_text = 'He throws webs'
# search_text = 'He is a super-soldier'
search_text = 'He is a super-soldier and hrows webs' 
search_vector = encoder.encode(search_text)
search_vector_as_array = np.array([search_vector])
faiss.normalize_L2(search_vector_as_array)

Perform a search within the created index (calculation of the distances between the search text and the strings within the index).

In [29]:
k = l2_index.ntotal
distances, ann = l2_index.search(search_vector_as_array, k=k)

Prepare the results to be displayed in a user-friendly format.

In [30]:
search_results = pd.DataFrame({'distances': distances[0], 'ann': ann[0]})
search_results.head()


,distances,ann
0,0.913094,3
1,1.180928,0
2,1.305882,1
3,1.314257,2


In [31]:
merged_df = pd.merge(search_results, df, left_on='ann', right_index=True)
merged_df.head()

,distances,ann,text,context
0,0.913094,3,A frail man enhanced to the peak of human phys...,captainamerica
1,1.180928,0,His secret identity is Peter Parker,spiderman
2,1.305882,1,A businessman and engineer who runs the compan...,ironman
3,1.314257,2,Superhuman spider-powers and abilities after b...,spiderman
